# 📊 Excel Viewer Pro (`xlsx-viewer-pro`)
### Headless Formula Engine (129+ Functions) & SIMD Spreadsheet Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminsk/xlsx_vievers/blob/master/notebooks/xlsx_viewer_quickstart.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/xlsx-viewer-pro.svg)](https://pypi.org/project/xlsx-viewer-pro/)
[![Conda-Forge](https://img.shields.io/conda/vn/conda-forge/xlsx-viewer-pro.svg)](https://anaconda.org/conda-forge/xlsx-viewer-pro)
[![Ubuntu / Debian PPA](https://img.shields.io/badge/Ubuntu%20%2F%20Debian-APT%20PPA-E95420?logo=ubuntu&logoColor=white)](https://eminsk.github.io/ppa/)
[![GitHub Stars](https://img.shields.io/badge/GitHub-eminsk%2Fxlsx__vievers-blue?logo=github)](https://github.com/eminsk/xlsx_vievers)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

**xlsx-viewer-pro** is a high-performance Python spreadsheet library, headless Excel formula engine (supporting **129+ functions** across Math, Finance, Lookups, Logic, Text, and Date/Time), hardware SIMD SSE2 math engine, and modern Office Ribbon desktop application.

### 🌟 Key Capabilities Demonstrated in this Notebook:
1. **Headless Formula Evaluation**: Evaluate complex Excel formulas directly in Python with zero GUI or Office installation.
2. **Financial Functions**: Real-time calculation of `PMT`, `PV`, `FV`, `NPV`, and `IRR`.
3. **Lookup & Reference Functions**: `XLOOKUP`, `VLOOKUP`, `HLOOKUP`, `INDEX`, and `MATCH`.
4. **Tabular Grid Evaluation**: In-memory 2D spreadsheet model with cross-cell and cross-sheet dependencies.
5. **CLI Utilities**: Instant terminal evaluation via `xlsx-viewer-pro --calc`.

## 1. 📦 Installation

Install `xlsx-viewer-pro` in Google Colab directly via `pip`:

> 💡 **For Ubuntu / Debian systems (outside Colab)**: You can also install system-wide via our official signed PPA:
> `curl -sS https://eminsk.github.io/ppa/setup.sh | sudo bash && sudo apt install -y python3-xlsx-viewer-pro`


In [ ]:
# 1. Cleanly remove legacy system package if previously installed:
!sudo apt-get remove -y python3-xlsx-viewer-pro >/dev/null 2>&1 || true

# 2. Install xlsx-viewer-pro from PyPI:
!pip install -q --no-cache-dir --ignore-installed xlsx-viewer-pro

# 3. Flush Python import cache:
import sys

for mod in list(sys.modules.keys()):
    if mod.startswith('xlsx_viewer'):
        del sys.modules[mod]

import xlsx_viewer

print('=' * 60)
print(f'✅ xlsx-viewer-pro v{xlsx_viewer.__version__} loaded successfully!')
print(f'📦 Package Origin: {xlsx_viewer.__file__}')
print(f'🐍 Python Runtime: {sys.version.split()[0]} ({sys.platform})')
print('=' * 60)


## 2. 🧮 Headless Formula Evaluation

Evaluate Excel formulas directly in Python without spinning up an Excel instance, COM object, or LibreOffice:

In [ ]:
from xlsx_viewer import evaluate_formula

# Math and Trigonometry
print('Arithmetic:    ', evaluate_formula('=SUM(10, 20, 30) * 2'))
print('Power & Root:  ', evaluate_formula('=ROUND(POWER(2, 10) + SQRT(144), 2)'))
print('Trigonometry:  ', evaluate_formula('=ROUND(SIN(PI() / 6) + COS(PI() / 3), 4)'))

# Financial: Loan Monthly Payment (PMT)
# $300,000 mortgage at 5.0% annual interest over 30 years (360 months)
monthly_pmt = evaluate_formula('=PMT(0.05 / 12, 360, -300000)')
print(f'Mortgage PMT:   ${monthly_pmt:,.2f} / month')

# Financial: Future Value of Investment (FV)
# Saving $500/month for 10 years (120 months) at 7.0% annual return
fv_savings = evaluate_formula('=FV(0.07 / 12, 120, -500)')
print(f'Investment FV:  ${fv_savings:,.2f}')

# Logic & String Manipulation
status = evaluate_formula('=IF(100 >= 80, "PASSED", "FAILED")')
print('Logic Result:  ', status)
greeting = evaluate_formula('=CONCAT("Invoice #", 1042, " - Total: ", TEXT(1250.5, "$#,##0.00"))')
print('Text Formatted:', greeting)


## 3. 🔍 Modern Lookup Functions (`XLOOKUP`, `VLOOKUP`, `INDEX/MATCH`)

Evaluate lookup formulas over structured 2D tabular data:

In [ ]:
from xlsx_viewer.formulas import FormulaEngine

# Create a sample employee salary table in-memory:
# Row 0: ID, Name, Department, Salary
# Row 1: 101, Alice, Engineering, 125000
# Row 2: 102, Bob, Data Science, 135000
# Row 3: 103, Charlie, DevOps, 115000
table_data = {
    (0, 0): 'ID',  (0, 1): 'Name',    (0, 2): 'Department',   (0, 3): 'Salary',
    (1, 0): 101,   (1, 1): 'Alice',   (1, 2): 'Engineering',  (1, 3): 125000,
    (2, 0): 102,   (2, 1): 'Bob',     (2, 2): 'Data Science', (2, 3): 135000,
    (3, 0): 103,   (3, 1): 'Charlie', (3, 2): 'DevOps',        (3, 3): 115000,
}

engine = FormulaEngine(lambda r, c, s=None: table_data.get((r, c), 0))

# 1. Classic VLOOKUP
vlookup_res = engine.evaluate('=VLOOKUP(102, A1:D4, 2, FALSE)')
print('VLOOKUP (ID 102 Name):       ', vlookup_res)

# 2. Modern XLOOKUP
xlookup_dept = engine.evaluate('=XLOOKUP(103, A2:A4, C2:C4)')
print('XLOOKUP (ID 103 Dept):       ', xlookup_dept)

# 3. Statistical Aggregate Formulas
avg_salary = engine.evaluate('=AVERAGE(D2:D4)')
max_salary = engine.evaluate('=MAX(D2:D4)')
total_headcount = engine.evaluate('=COUNT(A2:A4)')

print(f'Total Staff:                  {total_headcount}')
print(f'Average Salary:               ${avg_salary:,.2f}')
print(f'Maximum Salary:               ${max_salary:,.2f}')


## 4. ⚡ Command-Line Interface (CLI)

`xlsx-viewer-pro` provides an instant command-line utility for evaluating spreadsheet expressions directly in the shell:

In [ ]:
# Evaluate an expression from shell:
!xlsx-viewer-pro --calc "=PMT(0.06 / 12, 240, -250000)"

# Display supported formula functions:
!xlsx-viewer-pro --list-formulas | head -n 35


## 5. Summary & Resources

- **GitHub Repository**: [eminsk/xlsx_vievers](https://github.com/eminsk/xlsx_vievers)
- **PyPI Package**: [`pip install xlsx-viewer-pro`](https://pypi.org/project/xlsx-viewer-pro/)
- **Conda-Forge Package**: [`conda install -c conda-forge xlsx-viewer-pro`](https://anaconda.org/conda-forge/xlsx-viewer-pro)
- **Ubuntu / Debian APT PPA**: [`https://eminsk.github.io/ppa/`](https://eminsk.github.io/ppa/)
